# Tree rendering example

This notebook builds a synthetic budget and renders both ASCII and tikz/forest output.

In [ ]:
from pathlib import Path
import shutil
import pandas as pd
from budgie.tree import build_tree, render_ascii, render_tikz, display_tree

In [ ]:
table = pd.DataFrame([
    {"Name": "coherent_1", "Allocation": 4.0e-9, "CBE": 3.0e-9, "Type": "Static, Coherent", "Description": "first coherent term", "CBE Trace": "trace-a"},
    {"Name": "coherent_2", "Allocation": 4.0e-9, "CBE": 4.0e-9, "Type": "Static, Coherent", "Description": "second coherent term", "CBE Trace": "trace-b"},
    {"Name": "incoherent_1", "Allocation": 0.5e-9, "CBE": 1.0e-9, "Type": "Static, Incoherent", "Description": "incoherent term", "CBE Trace": "trace-c"},
    {"Name": "dynamic_1", "Allocation": 2.0e-9, "CBE": 2.0e-9, "Type": "Dynamic", "Description": "dynamic term", "CBE Trace": "trace-d"},
])
config = {
    "pp_gain": 0.1,
    "post_processing_chain": [
        {"op": "rss", "label": "Total raw value", "op_label": "RSS"},
        {"op": "scalar_multiply", "factor_key": "pp_gain", "label": "Post-processed value", "op_label": r"$\\times g_{pp}$"},
        {"op": "scalar_multiply", "factor": 5, "label": "5σ Post-processed value", "op_label": "5×"},
    ],
}
node = build_tree(table, config=config)

In [ ]:
print(render_ascii(node, show="both"))

## Colour coding

Colour coding has two independent parts. See
[`tree_rendering.md`](tree_rendering.md) for the full reference.

**1. Fill colour = `Type`** (emitted by `_tikz_style_block`). Every node is
filled by its `Type`, cycling through a fixed 7-colour palette (`blue!15`,
`green!15`, `orange!20`, `purple!15`, `teal!15`, `gray!20`, `cyan!15`).
Assignment is **positional** (pre-order walk, `index % 7`), so colours are *not*
stable across separate renders and an 8th type reuses the 1st type's colour.

**2. Allocation cues** (emitted by `render_tikz`'s label builders):

- Over-allocated leaf (`CBE > Allocation`): red bold border, `△!` prefix, and
  the CBE value in `\colorbox{yellow!50}`. Only when `alert_on_exceedances=True`
  (the default).
- Within-allocation leaf (`CBE <= Allocation`): CBE value underlined; not
  affected by `alert_on_exceedances`.
- Both apply to **leaves only** — subtotals and roll-ups just carry their fill.

The exposure-time calculator in `schmidt_ESP_template` uses **part 1 only**: it
builds its own node labels, so none of the part-2 cues appear in its figures.
`render_ascii` is plain text with **no** colour coding.


In [ ]:
tex = render_tikz(node, show="both", standalone=True)
if shutil.which("pdflatex"):
    display_tree(node, show="both")
else:
    print(tex)

In [ ]:
forest_tex = render_tikz(node, show="both", standalone=True, layout="forest")
outline_tex = render_tikz(node, show="both", standalone=True, layout="outline")

Path("/tmp/budgie_tree_forest.tex").write_text(forest_tex, encoding="utf-8")
Path("/tmp/budgie_tree_outline.tex").write_text(outline_tex, encoding="utf-8")

if shutil.which("pdflatex"):
    display_tree(node, show="both", layout="forest")
    display_tree(node, show="both", layout="outline")
else:
    print(forest_tex)
    print(outline_tex)


In [ ]:
# Same tree with negative alert cues suppressed: Type fills and the
# within-allocation underline remain; the red border / triangle / yellow
# highlight for over-allocated leaves are removed.
plain_tex = render_tikz(node, show="both", standalone=True, alert_on_exceedances=False)
if shutil.which("pdflatex"):
    display_tree(node, show="both", alert_on_exceedances=False)
else:
    print(plain_tex)
